<a href="https://colab.research.google.com/github/Rivera-Salvador/proyecto_LOGIEXPRESS-ENVIOS_caso9/blob/main/Modelo_segmentacion_Cliente_KMeans.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="text-align: center; max-width: 700px; margin: 0 auto; padding: 20px;">
  
  <h1>Análisis y Segmentación de Clientes de LogiExpress Mediante el Algoritmo K-Means para la Optimización del Servicio de Mensajería Urbana</h1>
  
  <img src="https://drive.google.com/uc?export=view&id=1DboIjZ06NSzY4z7paf9bZozu2BiGqA_s" width="250" style="margin: 15px 0;">
  
  <h1>CASO DE ESTUDIO PMD1: LOGIEXPRESS_ENVIOS</h1>
  
  <a href="https://github.com/Rivera-Salvador/proyecto_LOGIEXPRESS-ENVIOS_caso9/blob/main/LOGIEXPRESS_ENVIOS_01.ipynb">Ir al proyecto de LOGIEXPRESS-ENVIOS CASO-9 en github</a>

</div>

## 1 Descripción del problema de negocio

Contexto del negocio logiExpress es una empresa de mensajería urbana que gestiona documentos, paquetes pequeños, envíos urgentes y servicios especiales para clientes particulares y empresas. La operación diaria genera registros de clientes, tipos de envío, sedes y envíos realizados. La información permite analizar costos, pesos y características del servicio, pero requiere revisión porque se observan formatos mezclados, valores numéricos guardados como texto, pesos incompletos, nombres con espacios y registros que podrían haberse duplicado durante la atención.

## 2. Realizando la coneccion a drive para acceder a data set de formato csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

##3. Importando librerias de python

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_samples, silhouette_score
sns.set()
from sklearn.metrics import pairwise_distances_argmin_min

%matplotlib inline
from mpl_toolkits.mplot3d import Axes3D
plt.rcParams['figure.figsize'] = (16, 9)
plt.style.use('ggplot')

Utilizará ***KneeLocator*** para hallar el valor por defecto de k-means y dibujará las líneas automáticamente





In [ ]:
!pip install kneed

leemos df_original_copy_limpio.csv en pandas, para visualizar la data

In [ ]:
DRIVE_PROJECT_PATH = '/content/drive/MyDrive/Fundamento_gestion_datos/Proyecto_grupal/Modelo_Aprendizaje_No_Supervizado/'
df=pd.read_csv(f'{DRIVE_PROJECT_PATH}df_original_copy_limpio.csv')
display(df.head(10))

## 4. Separamos la data para entrenar en K-Means

Primero, transformamos el dataset de un registro por envío a un registro único por cliente, calculando sus métricas clave.

Las variables finales seleccionadas garantizan que el algoritmo K-Means no solo evalúe la capacidad financiera del cliente (costo_envio y descuento_pct), sino también su comportamiento logístico estructural (peso_paquete_kg, cantidad_paquetes y frecuencia_envios).
- La variable redundante como tipos_envio_tarifa_base fue excluida deliberadamente de la matriz de entrenamiento $X$. Dado que la tarifa base es un componente intrínseco y predeterminado del costo final del servicio, su inclusión junto a costo_envio habría generado un problema de multicolinealidad (redundancia de datos). Para garantizar que el algoritmo K-Means evalúe el comportamiento real del cliente

- el conjunto de datos se encuentra completamente preparado para ingresar a la fase de Estandarización de Variables y posterior evaluación del Número Óptimo de Clústeres (K).

**frecuencia_envios** es eltotal  de conteo de órdenes generadas por el cliente, sirviendo como indicador de retención y lealtad.

**costo_envio** Valor medio invertido por el cliente por cada operación realizada.

Impacto Logístico y Operativo (peso_paquete_kg y cantidad_paquetes) es el promedio de masa y volumen físico transportado, lo cual determina las necesidades de asignación de flota (motos,camionetas o treiler de carga).

**descuento_pct** es el nivel de respuesta promedio ante campañas de descuento y rebajas de tarifas.

**Métricas Categóricas**: Para variables como el canal (Web, App, Presencial) o el distrito, el código utiliza una función lambda para quedarse con la moda (x.mode()[0]), es decir, la opción que el cliente usa más seguido.

In [ ]:

# 1. Agrupamos datos a nivel de negocio (por cliente)
df_clientes_perfil = df.groupby('clientes_nombre').agg({
    'id_envio': 'count',                 # cuantos Id de envio tienen asignado a ese cliente
    'costo_envio': 'mean',               # Cuánto dinero deja en promedio por envío (Rentabilidad)
    'peso_paquete_kg': 'mean',           # Peso promedio (Impacto logístico en transporte)
    'cantidad_paquetes': 'mean',         # Volumen promedio por envío
    'descuento_pct': 'mean',             # Sensibilidad a promociones

    # Conservamos las categóricas más comunes de cada cliente para el análisis posterior
    'canal': lambda x: x.mode()[0] if not x.mode().empty else 'Desconocido',
    'clientes_distrito': lambda x: x.mode()[0] if not x.mode().empty else 'Desconocido',
    'clientes_segmento': lambda x: x.mode()[0] if not x.mode().empty else 'Desconocido'
}).rename(columns={'id_envio': 'frecuencia_envios'})# renombramos id_invio por frecuencia de envio

# 2. Seleccionar SOLO las variables numéricas de comportamiento para el algoritmo K-Means
variables_kmeans = ['frecuencia_envios', 'costo_envio', 'peso_paquete_kg', 'cantidad_paquetes', 'descuento_pct']
X = df_clientes_perfil[variables_kmeans]

print(f"Dataset listo para segmentar. Total de clientes únicos: {X.shape[0]}")

In [ ]:
df_clientes_perfil

In [ ]:
X

## 5. Estandarizacion de Variables con StandardScaler() de Scikit-Learn.

Antes de aplicar el algoritmo de agrupamiento K-Means y de evaluar la cantidad óptima de claster mediante el Método de la Curva de Codo, primero hay que realizar la  estandarización utilizando la herramienta StandardScaler() de Scikit-Learn.
- El algoritmo K-Means es un modelo basado en distancias geométricas (Euclidianas) para determinar qué tan similares o diferentes son los clientes entre sí. Por ende, es altamente sensible a las escalas y magnitudes de las variables del dataset

### Diagnostico
- En nuestros datos consolidados, la variable costo_envio maneja magnitudes que superan los 100.00 (en moneda), mientras que variables operativas críticas como cantidad_paquetes o la frecuencia_envios se mueven en rangos muy pequeños (valores típicamente entre 1 y 10)

- Evitar el Sesgo, si calculáramos las distancias matemáticas o aplicamos la Curva de Codo con los datos sin normalizar, el rango del factor monetario (costo_envio) dominaría por completo. El algoritmo asumiría erróneamente que las diferencias en el dinero son miles de veces más importantes que la frecuencia de envíos o el impacto logístico del peso del paquete
- Al transformar los datos con StandardScaler(), cada variable se ajusta para que su media sea igual a 0 y su desviación estándar sea igual a 1

In [ ]:
# PASO CRÍTICO: Estandarizar ANTES de calcular la Inercia para la Curva de Codo
scaler = StandardScaler()
X_escalado = scaler.fit_transform(X)

In [ ]:
X_escalado

### 6. **`Elbow`**

Utilizamos la distancia al centroide. Luego, graficamos la curva que nos permite determinar el K óptimo.

In [ ]:
Sum_of_squared_distances = []
K = np.arange(2, 11) # Incluimos el 1 para mejor perspectiva y limitamos a 10 por tamaño de muestra

for k in K:
    # Agregamos init, random_state y n_init para asegurar que el codo no cambie al reiniciar
    km = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    km = km.fit(X_escalado)

    distancia_total = km.inertia_
    distancia_media = np.divide(distancia_total, X_escalado.shape[0])
    Sum_of_squared_distances.append(distancia_media)

In [ ]:

from kneed import KneeLocator

# 1. DETECCIÓN AUTOMÁTICA DEL K ÓPTIMO (Por defecto)
# Sabiendo que la inercia media va decreciendo de forma convexa:
kneedle = KneeLocator(K, Sum_of_squared_distances, curve="convex", direction="decreasing")
k_optimo = kneedle.elbow

# Obtener la distancia (eje Y) del valor óptimo detectado automáticamente
idx_optimo = np.where(K == k_optimo)[0][0]
distancia_optima = Sum_of_squared_distances[idx_optimo]

# 2. Configurar el tamaño del gráfico
plt.figure(figsize = (10, 7))

# 3. Graficar la línea principal y los puntos rojos genéricos
plt.plot(K, Sum_of_squared_distances, lw=3, color='royalblue', label='Inercia Media')
plt.scatter(K, Sum_of_squared_distances, s=55, c='r', zorder=5)

# 4. RESALTAR EL K ÓPTIMO AUTOMÁTICO EN VERDE
# Línea vertical dinámica en el codo detectado
plt.axvline(x=k_optimo, color='forestgreen', linestyle='--', lw=2, label=f'K Óptimo Detectado ({k_optimo})')
# Punto verde destacado
plt.scatter(k_optimo, distancia_optima, s=180, c='forestgreen', marker='o', zorder=6)

# 5. Títulos y etiquetas profesionales
plt.title('Método del Codo: Determinación Automática de Clústeres\n(Análisis de Segmentación de Clientes - LogiExpress)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Número de Clústeres (K)', fontsize=12)
plt.ylabel('Dispersión Promedio por Cliente (Inercia Media)', fontsize=12)

# Ajustes de legibilidad y leyenda
plt.xticks(K)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='upper right', fontsize=11)

plt.show()

7.  # **` silhouette `**




In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Creamos una lista donde iremos guardando los valores medios de silhouette
lista_sil = []

# Definimos el rango de K a testear para asegurar consistencia en el plot
K_range = np.arange(2, 14)

# Entrenamos un modelo para cada número de cluster que queremos testear
for k in K_range:
    # Definimos y entrenamos el modelo
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km = km.fit(X_escalado)

    # Tomamos las etiquetas
    etiquetas = km.labels_

    # Calculamos el silhouette
    valor_medio_sil = silhouette_score(X_escalado, etiquetas)
    lista_sil.append(valor_medio_sil)

# --- DETECCIÓN AUTOMÁTICA DEL K ÓPTIMO (Máximo Coeficiente) ---
idx_max = np.argmax(lista_sil)         # Encuentra el índice del valor más alto
k_optimo_sil = K_range[idx_max]        # Obtiene el K correspondiente en base a ese índice
valor_max_sil = lista_sil[idx_max]     # Obtiene el valor máximo de Silhouette

# --- CONFIGURACIÓN DEL GRÁFICO ---
plt.figure(figsize = (10,7))
plt.plot(K_range, lista_sil, lw=3, color='darkorange', label='Silhouette Score')
plt.scatter(K_range, lista_sil, s=55, c='r', zorder=5)

# RESALTAR EL K ÓPTIMO EN VERDE
plt.axvline(x=k_optimo_sil, color='forestgreen', linestyle='--', lw=2, label=f'K Óptimo Máximo ({k_optimo_sil})')
plt.scatter(k_optimo_sil, valor_max_sil, s=180, c='forestgreen', marker='o', zorder=6)
# --------------------------------------------------------------

# Títulos profesionales orientados al negocio
plt.title('Análisis del Coeficiente de Silhouette Medio\n(Validación de Estructura de Clústeres - LogiExpress)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Número de Clústeres (K)', fontsize=12)
plt.ylabel('Coeficiente Silhouette Medio', fontsize=12)

# Ajustes de legibilidad y diseño
plt.xticks(K_range) # Cambiado a K_range para que coincida con tus datos anteriores
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='upper right', fontsize=11)

plt.show()

**Con el siguiente código, generamos los gráficos silhouette para todas las instancias**. La linea punteada verde indica el valor medio.

In [ ]:

from matplotlib.patches import Ellipse

# 1. Dataset escalado real
X_std = X_escalado

# 2. Lista de K a evaluar
lista_k = [2, 3, 4, 5]

for i, k in enumerate(lista_k):
    fig, (ax1, ax2) = plt.subplots(1, 2)
    fig.set_size_inches(18, 7)

    # Instanciar y entrenar
    km = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    labels = km.fit_predict(X_std)
    centroids = km.cluster_centers_ # Corrected: Removed parentheses and added underscore

    # Gráfico 1: Silhouette (Barras por clúster)
    silhouette_vals = silhouette_samples(X_std, labels)
    y_lower, y_upper = 0, 0
    for j, cluster in enumerate(np.unique(labels)):
        cluster_silhouette_vals = silhouette_vals[labels == cluster]
        cluster_silhouette_vals.sort()
        y_upper += len(cluster_silhouette_vals)
        ax1.barh(range(y_lower, y_upper), cluster_silhouette_vals, edgecolor='none', height=1)
        ax1.text(-0.03, (y_lower + y_upper) / 2, f"Clúster {j}")
        y_lower += len(cluster_silhouette_vals)

    avg_score = np.mean(silhouette_vals)
    ax1.axvline(avg_score, linestyle='--', linewidth=2, color='green', label=f'Promedio: {avg_score:.3f}')
    ax1.set_yticks([])
    ax1.set_xlim([-0.1, 1])
    ax1.set_xlabel('Valores del Coeficiente de Silhouette')
    ax1.set_ylabel('Grupos / Clústeres')
    ax1.set_title('Grosor y Consistencia por Clúster', y=1.02)
    ax1.legend(loc='lower right')

    # Gráfico 2: Distribución Espacial con Círculos/Elipses contenedores
    # Usamos un mapa de colores ordenado para que coincida con las barras
    colores = plt.cm.tab10(np.linspace(0, 1, 10))

    for j in range(k):
        # Filtrar puntos del clúster actual en las dimensiones de Frecuencia (0) y Costo (1)
        puntos_cluster = X_std[labels == j]
        x_pts = puntos_cluster[:, 0]
        y_pts = puntos_cluster[:, 1]

        # Dibujar los puntos del grupo
        ax2.scatter(x_pts, y_pts, color=colores[j], s=50, alpha=0.85, label=f'Clientes Clúster {j}')

        # --- DIBUJAR EL CÍRCULO ENVOLVENTE ---
        if len(puntos_cluster) > 1:
            # Calcular el centroide real en 2D y la dispersión (covarianza) para definir el radio
            centro_x, centro_y = centroids[j, 0], centroids[j, 1]
            ancho = (x_pts.max() - x_pts.min()) * 1.2
            alto = (y_pts.max() - y_pts.min()) * 1.2

            # Crear la elipse contenedora alrededor del grupo
            elipse = Ellipse(xy=(centro_x, centro_y), width=ancho, height=alto,
                             edgecolor=colores[j], facecolor=colores[j], alpha=0.1, lw=2, linestyle='-')
            ax2.add_patch(elipse)

    # Dibujar los centroides como estrellas rojas
    ax2.scatter(centroids[:k, 0], centroids[:k, 1], marker='*', c='red', s=250, zorder=10, label='Centroides')

    # Límites dinámicos basados en la dispersión real de tus clientes
    ax2.set_xlim([X_std[:, 0].min() - 0.8, X_std[:, 0].max() + 0.8])
    ax2.set_ylim([X_std[:, 1].min() - 0.8, X_std[:, 1].max() + 0.8])

    ax2.set_xlabel('Frecuencia de Envíos (Estandarizada)')
    ax2.set_ylabel('Costo de Envío Promedio (Estandarizado)')
    ax2.set_title('Grupos Delimitados en el Espacio Financiero', y=1.02)
    ax2.legend(loc='upper right')

    plt.tight_layout()
    plt.suptitle(f'Análisis de Perfilamiento de Silhouette para LogiExpress (K = {k})',
                 fontsize=16, fontweight='semibold', y=1.05)
    plt.show()

## Sustento de la Decisión ($K = 3$)

- El Método del Codo detectó automáticamente un quiebre claro en $K = 3$. Esto significa que añadir un cuarto grupo no reduce la dispersión interna de forma significativa
- El Coeficiente de Silhouette Medio sugería matemáticamente $K = 2$ como el pico más alto. Sin embargo, el la gráficos de perfilamiento detallados (Silhouette Plots), con $K = 2$ terminas con grupos demasiado masivos y genéricos. Con $K = 3$, los clústeres mantienen un grosor equilibrado, superan la línea discontinua del promedio global y delimitan zonas compactas en el espacio de Frecuencia vs. Costo.

# Entrenando el Modelo

In [ ]:
# Entrenar el modelo final con el K elegido por el experto de negocio
kmeans_final = KMeans(n_clusters=3, init='k-means++', random_state=42, n_init=10)

# Guardar la asignación en tu DataFrame de perfiles
df_clientes_perfil['segmento_final'] = kmeans_final.fit_predict(X_escalado)

print("¡Clientes etiquetados con su segmento de negocio final (0, 1 o 2)!")

In [ ]:
def analizar_nuevo_cliente(frecuencia, costo, peso, cantidad, descuento):
    """
    Automatiza el ingreso de datos, estandariza según el modelo de LogiExpress,
    asigna el clúster óptimo y despliega el reporte de perfilamiento detallado
    con sugerencias avanzadas de tarifas y asignación de flotas logísticas.
    """
    # 1. Empaquetar los datos automáticamente en la estructura correcta
    datos_cliente = {
        'frecuencia_envios': [frecuencia],
        'costo_envio': [costo],
        'peso_paquete_kg': [peso],
        'cantidad_paquetes': [cantidad],
        'descuento_pct': [descuento]
    }

    # 2. Convertir a DataFrame usando las variables oficiales del proyecto
    df_nuevo = pd.DataFrame(datos_cliente)

    # 3. Estandarizar usando el scaler calibrado original (¡Sin fit!)
    nuevo_escalado = scaler.transform(df_nuevo)

    # 4. Predecir el clúster con el modelo KMeans final (K=3)
    cluster_asignado = kmeans_final.predict(nuevo_escalado)[0]

    # 5. Extraer promedios reales de la base de datos para comparar
    promedios_clusters = df_clientes_perfil.groupby('segmento_final')[variables_kmeans].mean()

    # 6. Desplegar la interfaz del Reporte Automatizado
    print("="*65)
    print(f"       📋 REPORTE AUTOMÁTICO DE PERFILAMIENTO DE CLIENTE")
    print("="*65)
    print(f"➔ CLÚSTER ASIGNADO POR EL MODELO: CLÚSTER {cluster_asignado}\n")
    print("📊 COMPARATIVA DE MÉTRICAS (Cliente vs. Promedios del Grupo):")
    print("-" * 65)

    for var in variables_kmeans:
        valor_cliente = datos_cliente[var][0]
        valor_promedio_grupo = promedios_clusters.loc[cluster_asignado, var]

        print(f"• {var.upper()}:")
        print(f"  - Valor Ingresado:       {valor_cliente:.2f}")
        print(f"  - Promedio de su Grupo:  {valor_promedio_grupo:.2f}")

        # Diagnóstico analítico de la métrica
        if valor_cliente > valor_promedio_grupo:
            print(f"  📢 Diagnóstico: Se encuentra POR ENCIMA de la media de su grupo.")
        else:
            print(f"  📢 Diagnóstico: Se encuentra POR DEBAJO o al nivel de la media de su grupo.")
        print("-" * 65)
    # 7. ESTRATEGIA COMERCIAL Y RECOMENDACIÓN DE TARIFAS POR FLOTA (VERSION CORREGIDA)
    print("\n💡 RECOMENDACIÓN ESTRATÉGICA DE NEGOCIO:")
    costo_promedio_grupo = promedios_clusters.loc[cluster_asignado, 'costo_envio']

    # REGLA DE INGENIERÍA LOGÍSTICA: Definir flota por los datos reales ingresados
    if peso > 25.0 or cantidad > 15:
        # SI EL PAQUETE ES PESADO O SON MUCHOS, VA EN CAMIONETA INDEPENDIENTE DEL CLÚSTER
        print("👉 PERFIL ASIGNADO: TRANSPORTE DE CARGA PESADA / VOLUMEN")
        print("  - Estrategia: Asegurar disponibilidad vehicular y optimizar la estiba de carga.")
        print("  - Flota Recomendada: Flota Pesada (Camionetas de reparto, Van o Camiones ligeros).")
        print(f"  - Sugerencia de Tarifa: El costo medio operativo para este volumen es de {costo_promedio_grupo:.2f}.")
        if costo < costo_promedio_grupo:
            print(f"    🚨 Alerta de Pérdida Logística: Cobrar {costo:.2f} no cubre los costos operativos de camioneta. Debes establecer un cobro mínimo de {costo_promedio_grupo:.2f}.")
        else:
            print(f"    ✅ Margen de Seguridad: Cobro adecuado ({costo:.2f}) para amortizar la flota pesada.")

    elif frecuencia >= 6:
        # SI EL CLIENTE TIENE ALTA FRECUENCIA, ES UN CORPORATIVO PREMIUM
        print("👉 PERFIL ASIGNADO: CORPORATIVO / PREMIUM (ALTA FRECUENCIA)")
        print("  - Estrategia: Asignar un ejecutivo de cuentas clave (KAM) y contratos de fidelización.")
        print("  - Flota Recomendada: Operación Mixta (Motos para urgencias + Camionetas programadas).")
        print(f"  - Sugerencia de Tarifa: El promedio de este segmento premium es {costo_promedio_grupo:.2f}.")
        if costo < costo_promedio_grupo:
            print(f"    ⚠️ Alerta de Sub-tarifa: Ajustar al alza hacia los {costo_promedio_grupo:.2f} con tarifa plana corporativa.")
        else:
            print(f"    ✅ Tarifa Correcta: Estás reteniendo un margen saludable.")

    else:
        # POR DESCARTE, ENVIOS LIGEROS Y OCASIONALES
        print("👉 PERFIL ASIGNADO: OPERATIVO / RECURRENTE ESTÁNDAR")
        print("  - Estrategia: Incentivar el uso exclusivo de la App y cupones automatizados.")
        print("  - Flota Recomendada: Flota Eficiente de Motos (Envíos ligeros de alta densidad).")
        print(f"  - Sugerencia de Tarifa: El costo base optimizado es de {costo_promedio_grupo:.2f}.")
        if costo > costo_promedio_grupo:
            print(f"    ⚠️ Riesgo de Churn (Fuga): Cobras {costo:.2f}, que supera la media de {costo_promedio_grupo:.2f}. Reduce la tarifa unitaria.")
        else:
            print(f"    ✅ Tarifa Competitiva: Cobro óptimo para mantener rutas densas en motos.")

    print("="*65)

## Clúster 1: Carga Pesada (Camiones / Camionetas).

In [ ]:
analizar_nuevo_cliente(1,150.00,120.5,45,0)

## Clúster 2: Perfil Operativo / Recurrente Estándar (Motos).

In [ ]:
analizar_nuevo_cliente(2,145.00,8.5,2,5)

## Clúster 0: Clientes Corporativos / Premium.

In [ ]:
analizar_nuevo_cliente(12,165.00,15.0,4,12)

In [ ]:
def analizar_nuevo_cliente(frecuencia, costo, peso, cantidad, descuento):
    """
    Automatiza el ingreso de datos, estandariza según el modelo de LogiExpress,
    asigna el clúster óptimo y despliega el reporte de perfilamiento detallado
    con sugerencias avanzadas de tarifas y asignación de flotas logísticas.
    """
    # 1. Empaquetar los datos automáticamente en la estructura correcta
    datos_cliente = {
        'frecuencia_envios': [frecuencia],
        'costo_envio': [costo],
        'peso_paquete_kg': [peso],
        'cantidad_paquetes': [cantidad],
        'descuento_pct': [descuento]
    }

    # 2. Convertir a DataFrame usando las variables oficiales del proyecto
    df_nuevo = pd.DataFrame(datos_cliente)

    # 3. Estandarizar usando el scaler calibrado original (¡Sin fit!)
    nuevo_escalado = scaler.transform(df_nuevo)

    # 4. Predecir el clúster con el modelo KMeans final (K=3)
    cluster_asignado = kmeans_final.predict(nuevo_escalado)[0]

    # 5. Extraer promedios reales de la base de datos para comparar
    promedios_clusters = df_clientes_perfil.groupby('segmento_final')[variables_kmeans].mean()

    # 6. Desplegar la interfaz del Reporte Automatizado
    print("="*65)
    print(f"       📋 REPORTE AUTOMÁTICO DE PERFILAMIENTO DE CLIENTE")
    print("="*65)
    print(f"➔ CLÚSTER ASIGNADO POR EL MODELO: CLÚSTER {cluster_asignado}\n")
    print("📊 COMPARATIVA DE MÉTRICAS (Cliente vs. Promedios del Grupo):")
    print("-" * 65)

    for var in variables_kmeans:
        valor_cliente = datos_cliente[var][0]
        valor_promedio_grupo = promedios_clusters.loc[cluster_asignado, var]

        print(f"• {var.upper()}:")
        print(f"  - Valor Ingresado:       {valor_cliente:.2f}")
        print(f"  - Promedio de su Grupo:  {valor_promedio_grupo:.2f}")

        # Diagnóstico analítico de la métrica
        if valor_cliente > valor_promedio_grupo:
            print(f"  📢 Diagnóstico: Se encuentra POR ENCIMA de la media de su grupo.")
        else:
            print(f"  📢 Diagnóstico: Se encuentra POR DEBAJO o al nivel de la media de su grupo.")
        print("-" * 65)
    # 7. ESTRATEGIA COMERCIAL Y RECOMENDACIÓN DE TARIFAS POR FLOTA (VERSION CORREGIDA)
    print("\n💡 RECOMENDACIÓN ESTRATÉGICA DE NEGOCIO:")
    costo_promedio_grupo = promedios_clusters.loc[cluster_asignado, 'costo_envio']

    # REGLA DE INGENIERÍA LOGÍSTICA: Definir flota por los datos reales ingresados
    if peso > 25.0 or cantidad > 15:
        # SI EL PAQUETE ES PESADO O SON MUCHOS, VA EN CAMIONETA INDEPENDIENTE DEL CLÚSTER
        print("👉 PERFIL ASIGNADO: TRANSPORTE DE CARGA PESADA / VOLUMEN")
        print("  - Estrategia: Asegurar disponibilidad vehicular y optimizar la estiba de carga.")
        print("  - Flota Recomendada: Flota Pesada (Camionetas de reparto, Van o Camiones ligeros).")
        print(f"  - Sugerencia de Tarifa: El costo medio operativo para este volumen es de {costo_promedio_grupo:.2f}.")
        if costo < costo_promedio_grupo:
            print(f"    🚨 Alerta de Pérdida Logística: Cobrar {costo:.2f} no cubre los costos operativos de camioneta. Debes establecer un cobro mínimo de {costo_promedio_grupo:.2f}.")
        else:
            print(f"    ✅ Margen de Seguridad: Cobro adecuado ({costo:.2f}) para amortizar la flota pesada.")

    elif frecuencia >= 6:
        # SI EL CLIENTE TIENE ALTA FRECUENCIA, ES UN CORPORATIVO PREMIUM
        print("👉 PERFIL ASIGNADO: CORPORATIVO / PREMIUM (ALTA FRECUENCIA)")
        print("  - Estrategia: Asignar un ejecutivo de cuentas clave (KAM) y contratos de fidelización.")
        print("  - Flota Recomendada: Operación Mixta (Motos para urgencias + Camionetas programadas).")
        print(f"  - Sugerencia de Tarifa: El promedio de este segmento premium es {costo_promedio_grupo:.2f}.")
        if costo < costo_promedio_grupo:
            print(f"    ⚠️ Alerta de Sub-tarifa: Ajustar al alza hacia los {costo_promedio_grupo:.2f} con tarifa plana corporativa.")
        else:
            print(f"    ✅ Tarifa Correcta: Estás reteniendo un margen saludable.")

    else:
        # POR DESCARTE, ENVIOS LIGEROS Y OCASIONALES
        print("👉 PERFIL ASIGNADO: OPERATIVO / RECURRENTE ESTÁNDAR")
        print("  - Estrategia: Incentivar el uso exclusivo de la App y cupones automatizados.")
        print("  - Flota Recomendada: Flota Eficiente de Motos (Envíos ligeros de alta densidad).")
        print(f"  - Sugerencia de Tarifa: El costo base optimizado es de {costo_promedio_grupo:.2f}.")
        if costo > costo_promedio_grupo:
            print(f"    ⚠️ Riesgo de Churn (Fuga): Cobras {costo:.2f}, que supera la media de {costo_promedio_grupo:.2f}. Reduce la tarifa unitaria.")
        else:
            print(f"    ✅ Tarifa Competitiva: Cobro óptimo para mantener rutas densas en motos.")

    print("="*65)

In [ ]:

def analizar_nuevo_cliente(frecuencia, costo, peso, cantidad, descuento):
    """
    Automatiza el ingreso de datos, estandariza según el modelo de LogiExpress,
    asigna el clúster óptimo y despliega el reporte de perfilamiento detallado
    con sugerencias avanzadas de tarifas y asignación de flotas logísticas.
    """
    # 1. Empaquetar los datos automáticamente en la estructura correcta
    datos_cliente = {
        'frecuencia_envios': [frecuencia],
        'costo_envio': [costo],
        'peso_paquete_kg': [peso],
        'cantidad_paquetes': [cantidad],
        'descuento_pct': [descuento]
    }

    # 2. Convertir a DataFrame usando las variables oficiales del proyecto
    df_nuevo = pd.DataFrame(datos_cliente)

    # 3. Estandarizar usando el scaler calibrado original (¡Sin fit!)
    nuevo_escalado = scaler.transform(df_nuevo)

    # 4. Predecir el clúster con el modelo KMeans final (K=3)
    cluster_asignado = kmeans_final.predict(nuevo_escalado)[0]

    # 5. Extraer promedios reales de la base de datos para comparar
    promedios_clusters = df_clientes_perfil.groupby('segmento_final')[variables_kmeans].mean()

    # 6. Desplegar la interfaz del Reporte Automatizado
    print("="*65)
    print(f"       📋 REPORTE AUTOMÁTICO DE PERFILAMIENTO DE CLIENTE")
    print("="*65)
    print(f"➔ CLÚSTER ASIGNADO POR EL MODELO: CLÚSTER {cluster_asignado}\n")
    print("📊 COMPARATIVA DE MÉTRICAS (Cliente vs. Promedios del Grupo):")
    print("-" * 65)

    for var in variables_kmeans:
        valor_cliente = datos_cliente[var][0]
        valor_promedio_grupo = promedios_clusters.loc[cluster_asignado, var]

        print(f"• {var.upper()}:")
        print(f"  - Valor Ingresado:       {valor_cliente:.2f}")
        print(f"  - Promedio de su Grupo:  {valor_promedio_grupo:.2f}")

        # Diagnóstico analítico de la métrica
        if valor_cliente > valor_promedio_grupo:
            print(f"  📢 Diagnóstico: Se encuentra POR ENCIMA de la media de su grupo.")
        else:
            print(f"  📢 Diagnóstico: Se encuentra POR DEBAJO o al nivel de la media de su grupo.")
        print("-" * 65)
# =================================================================
    # 7. PIPELINE DE PRICING LOGÍSTICO Y AJUSTE DINÁMICO DE TARIFAS
    # =================================================================
    print("\n💡 EVALUACIÓN DEL PIPELINE DE PRECIOS (PRICING LOGÍSTICO):")

    # Extracción de la media histórica del costo para el clúster asignado
    costo_medio_historico = promedios_clusters.loc[cluster_asignado, 'costo_envio']

    # REGLA DE NEGOCIO: Definición del Umbral de Margen Operativo (Ej: 95% del promedio)
    # Si cobramos menos del 95% de la media histórica, entramos en zona de pérdida operativa.
    UMBRAL_MARGEN = 0.95
    tarifa_minima_garantizada = costo_medio_historico * UMBRAL_MARGEN

    if cluster_asignado == 1:  # Tu Clúster Real de Carga Pesada (Camiones/Camionetas)
        print("👉 PERFIL EVALUADO: TRANSPORTE DE CARGA PESADA / VOLUMEN (FLOTA PESADA)")
        print(f"  - Costo Medio Histórico del Grupo: {costo_medio_historico:.2f}")
        print(f"  - Tarifa Mínima para Garantizar Margen: {tarifa_minima_garantizada:.2f}")
        print("-" * 65)

        # APLICACIÓN DE LA REGLA DE SUB-TARIFA
        if costo < tarifa_minima_garantizada:
            recargo_peso_muerto = tarifa_minima_garantizada - costo
            print(f"    🚨 ALERTA DETECTADA: [SUB-TARIFA] - RIESGO OPERATIVO")
            print(f"    📢 Diagnóstico Técnico: El costo ingresado ({costo:.2f}) está por debajo de la media histórica.")
            print(f"    💡 Acción Sugerida: Aplicar de forma inmediata:")
            print(f"       1. Un RECARGO POR PESO MUERTO de: +{recargo_peso_muerto:.2f}")
            print(f"       2. O forzar un COBRO MÍNIMO DE SEGURIDAD de: {tarifa_minima_garantizada:.2f}")
            print(f"    ⚠️ Nota: Esto garantiza cubrir los costos fijos de combustible y depreciación de los vehículos grandes.")
        else:
            print(f"    ✅ MARGEN DE SEGURIDAD VALIDADO: El cobro actual de {costo:.2f} absorbe correctamente los costos operativos de la flota pesada.")

    elif cluster_asignado == 0:  # Perfil Corporativo
        print("👉 PERFIL EVALUADO: CORPORATIVO / PREMIUM")
        if costo < costo_medio_historico:
            print(f"    ⚠️ SUGERENCIA: Ajustar tarifa hacia la media de {costo_medio_historico:.2f} mediante contratos de tarifa plana por volumen.")
        else:
            print(f"    ✅ RENTABILIDAD ALTA: Cliente reteniendo márgenes óptimos.")

    elif cluster_asignado == 2:  # Tu Clúster Real de Motos
        print("👉 PERFIL EVALUADO: OPERATIVO / RECURRENTE (MOTOS)")
        if costo > costo_medio_historico:
            print(f"    ⚠️ ALERTA DE FUGA (CHURN): Estás cobrando {costo:.2f} (Media: {costo_medio_historico:.2f}). Reduce el precio unitario para proteger la densidad de rutas.")
        else:
            print(f"    ✅ TARIFA COMPETITIVA: Precio ideal para fidelización en la App.")

    print("="*65)

In [ ]:
analizar_nuevo_cliente(1,150.00,120.5,45,0)